# 04 — Held-Out Evaluation

**Where this picks up.** `03_uplift_models.ipynb` compared several uplift models on `train`/`val` only; `test.parquet` is used here for the first and only time, to produce this project's final numbers — Qini AUC and uplift@k with bootstrap confidence intervals, Qini curves, and a business-framed targeting-impact translation. All metrics come from `src/evaluation.py`; this notebook only composes and interprets them.

**Scope.** Neither `02` nor `03` persists fitted models to disk, so this notebook refits the four core meta-learners (S-Learner GBM/Logistic, T-Learner GBM, X-Learner GBM) on `train.parquet`, using the same tuning *procedure* as `03` Step 4 — not necessarily its exact output (see note in Step 1). The `03` Step 8–11 baselines (Decision Tree, Random Forest, MLP, TARNet) are out of scope: they would need a model-persistence step added to `03` first, so they aren't refit from scratch here.

**No leakage.** `test.parquet` is loaded but never passed to `.fit()` — only to `.predict()` and to `src/evaluation.py`.

In [1]:
%pip install pandas numpy scikit-learn scikit-uplift pyarrow plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import warnings



project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))

from src.models import (
    XLearner,
    build_gradient_boosting_base_learner,
    build_gradient_boosting_effect_regressor,
    build_logistic_regression_base_learner,
    collect_uplift_predictions,
    prepare_arm_splits,
    tune_base_learner_hyperparameters,
)
from src.evaluation import (
    compare_models_on_test,
    compute_qini_curve,
    compute_targeting_impact,
    evaluate_uplift_model,
    plot_qini_curve,
    plot_qini_curves_comparison, 
    qini_auc_with_ci,
    uplift_at_k_with_ci,
)

from sklift.models import SoloModel, TwoModels

pd.set_option("display.max_columns", None)

with open("../data/processed/feature_manifest.json") as f:
    manifest = json.load(f)

feature_columns = manifest["feature_columns"]
treatment_col = manifest["treatment_col"]

train_df = pd.read_parquet("../data/processed/train.parquet")
val_df = pd.read_parquet("../data/processed/val.parquet")
test_df = pd.read_parquet("../data/processed/test.parquet")

print(f"train: {train_df.shape}, val: {val_df.shape}, test: {test_df.shape}")
print(f"Feature columns ({len(feature_columns)}): {feature_columns}")
print("test.parquet loaded now, for the first time in this project, for final evaluation only.")

train: (38400, 13), val: (12800, 13), test: (12800, 13)
Feature columns (9): ['recency', 'history', 'mens', 'womens', 'newbie', 'zip_code_Surburban', 'zip_code_Urban', 'channel_Phone', 'channel_Web']
test.parquet loaded now, for the first time in this project, for final evaluation only.


## Step 1: Refit the core models on train+val, evaluate once on test

Hyperparameters are re-tuned here rather than reusing `03`'s exact config, since this is an independent search over the same space and scoring metric, not a replay. 

**Note:** the result differs from `03`'s Step 4 output only in `min_samples_leaf` (100 vs 50); `max_iter` matches (150). Models are refit on train+val combined (not train alone), since test is now the only held-out set and every available non-test row should inform the final fit.

In [3]:
gbm_param_distributions = {
    "max_iter": [50, 100, 150],
    "max_depth": [3, 5, 7, None],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_samples_leaf": [20, 50, 100],
}

tuned_gbm = tune_base_learner_hyperparameters(
    estimator=build_gradient_boosting_base_learner(),
    param_distributions=gbm_param_distributions,
    X=train_df[feature_columns],
    y=train_df["visit"],
    n_iter=15,
    cv=5,
    scoring="average_precision",
    random_state=42,
)
gbm_params = {
    key: value
    for key, value in tuned_gbm.get_params().items()
    if key in {"max_iter", "max_depth", "learning_rate", "min_samples_leaf"}
}

logreg_param_distributions = {"classifier__C": np.logspace(-3, 2, 6)}
tuned_logreg_pipeline = tune_base_learner_hyperparameters(
    estimator=build_logistic_regression_base_learner(numeric_features=["recency", "history"]),
    param_distributions=logreg_param_distributions,
    X=train_df[feature_columns],
    y=train_df["visit"],
    n_iter=6,
    cv=5,
    scoring="average_precision",
    random_state=42,
)
best_C = tuned_logreg_pipeline.named_steps["classifier"].C

print("Tuned GBM config:", gbm_params)
print("Tuned LogisticRegression C:", best_C)

Tuned GBM config: {'learning_rate': 0.1, 'max_depth': 3, 'max_iter': 150, 'min_samples_leaf': 100}
Tuned LogisticRegression C: 100.0


In [4]:
def fit_core_models(train_X, train_y, train_treatment, gbm_params, best_C):
    """Fit this notebook's four core meta-learners for one arm.

    Notebook-local helper (not in src/models.py), mirroring
    03_uplift_models.ipynb's fit_candidate_models: it encodes this
    notebook's own scope decision (which models to evaluate on test.parquet
    -- see the markdown note above), not a generic reusable primitive. The
    building blocks it calls (base-learner factories, XLearner) live in
    src/models.py.
    """
    models = {}

    models["S-Learner (GBM)"] = SoloModel(
        estimator=build_gradient_boosting_base_learner(**gbm_params, random_state=42)
    )
    models["S-Learner (GBM)"].fit(train_X, train_y, train_treatment)

    models["S-Learner (Logistic)"] = SoloModel(
        estimator=build_logistic_regression_base_learner(
            numeric_features=["recency", "history"], C=best_C, random_state=42
        )
    )
    models["S-Learner (Logistic)"].fit(train_X, train_y, train_treatment)

    models["T-Learner (GBM)"] = TwoModels(
        estimator_trmnt=build_gradient_boosting_base_learner(**gbm_params, random_state=42),
        estimator_ctrl=build_gradient_boosting_base_learner(**gbm_params, random_state=42),
        method="vanilla",
    )
    models["T-Learner (GBM)"].fit(train_X, train_y, train_treatment)

    models["X-Learner (GBM)"] = XLearner(
        estimator_outcome_treatment=build_gradient_boosting_base_learner(**gbm_params, random_state=42),
        estimator_outcome_control=build_gradient_boosting_base_learner(**gbm_params, random_state=42),
        estimator_effect_treatment=build_gradient_boosting_effect_regressor(**gbm_params, random_state=42),
        estimator_effect_control=build_gradient_boosting_effect_regressor(**gbm_params, random_state=42),
        propensity=None,
    )
    models["X-Learner (GBM)"].fit(train_X, train_y, train_treatment)

    return models

In [ ]:
mens_splits = prepare_arm_splits(
    train_df, val_df, test_df,
    treatment_group="Mens E-Mail",
    feature_columns=feature_columns,
    outcome_col="visit",
)

# Final refit uses train+val combined, not train alone: val's model-selection
# role ended in 03_uplift_models.ipynb, so every non-test row should inform
# the model actually evaluated on test.parquet, per the markdown above.
X_train_mens = pd.concat([mens_splits["train"][0], mens_splits["val"][0]], ignore_index=True)
y_train_mens = pd.concat([mens_splits["train"][1], mens_splits["val"][1]], ignore_index=True)
treat_train_mens = pd.concat([mens_splits["train"][2], mens_splits["val"][2]], ignore_index=True)
X_test_mens, y_test_mens, treat_test_mens = mens_splits["test"]

mens_models = fit_core_models(X_train_mens, y_train_mens, treat_train_mens, gbm_params, best_C)
print(f"Fitted models for Mens E-Mail vs No E-Mail on train+val (n={len(X_train_mens)}):", list(mens_models.keys()))

Fitted models for Mens E-Mail vs No E-Mail on train+val (n=34090): ['S-Learner (GBM)', 'S-Learner (Logistic)', 'T-Learner (GBM)', 'X-Learner (GBM)']


In [6]:
womens_splits = prepare_arm_splits(
    train_df, val_df, test_df,
    treatment_group="Womens E-Mail",
    feature_columns=feature_columns,
    outcome_col="visit",
)

X_train_womens = pd.concat([womens_splits["train"][0], womens_splits["val"][0]], ignore_index=True)
y_train_womens = pd.concat([womens_splits["train"][1], womens_splits["val"][1]], ignore_index=True)
treat_train_womens = pd.concat([womens_splits["train"][2], womens_splits["val"][2]], ignore_index=True)
X_test_womens, y_test_womens, treat_test_womens = womens_splits["test"]

womens_models = fit_core_models(X_train_womens, y_train_womens, treat_train_womens, gbm_params, best_C)
print(f"Fitted models for Womens E-Mail vs No E-Mail on train+val (n={len(X_train_womens)}):", list(womens_models.keys()))

Fitted models for Womens E-Mail vs No E-Mail on train+val (n=34155): ['S-Learner (GBM)', 'S-Learner (Logistic)', 'T-Learner (GBM)', 'X-Learner (GBM)']


## Step 2: Evaluate every model on `test.parquet`

`evaluate_uplift_model` (from `src/evaluation.py`) bundles Qini AUC with a bootstrap CI, `uplift_at_k` with a bootstrap CI at 10/20/30% targeting fractions, and a Qini curve, in one call per model. `n_bootstrap=1000` and `random_state=42` are fixed across every model within an arm so every model's confidence interval is built from the same resampling draws, keeping the comparison apples-to-apples.

In [7]:
mens_test_predictions = collect_uplift_predictions(mens_models, X_test_mens)
womens_test_predictions = collect_uplift_predictions(womens_models, X_test_womens)

mens_results = {
    model_name: evaluate_uplift_model(
        y_test_mens, mens_test_predictions[model_name], treat_test_mens,
        k_values=(0.10, 0.20, 0.30), n_bootstrap=1000, ci_level=0.95, random_state=42,
    )
    for model_name in mens_models
}

womens_results = {
    model_name: evaluate_uplift_model(
        y_test_womens, womens_test_predictions[model_name], treat_test_womens,
        k_values=(0.10, 0.20, 0.30), n_bootstrap=1000, ci_level=0.95, random_state=42,
    )
    for model_name in womens_models
}

print("Evaluation complete for both arms (Mens E-Mail, Womens E-Mail) on test.parquet.")

/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.
  warnings.warn(msg, category=FutureWarning)
/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.
  warnings.warn(msg, category=FutureWarning)
/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly i

Evaluation complete for both arms (Mens E-Mail, Womens E-Mail) on test.parquet.


In [8]:
mens_comparison_df = compare_models_on_test(mens_results)
print("--- Mens E-Mail: test-set comparison (sorted by Qini AUC) ---")
display(mens_comparison_df)

--- Mens E-Mail: test-set comparison (sorted by Qini AUC) ---


,model,qini_auc,qini_auc_ci_lower,qini_auc_ci_upper,uplift_at_10pct,uplift_at_10pct_ci_lower,uplift_at_10pct_ci_upper,uplift_at_20pct,uplift_at_20pct_ci_lower,uplift_at_20pct_ci_upper,uplift_at_30pct,uplift_at_30pct_ci_lower,uplift_at_30pct_ci_upper,n_samples
0,S-Learner (GBM),0.012026,-0.020695,0.047703,0.111710,0.053809,0.168581,0.085196,0.046602,0.122396,0.075923,0.048627,0.107399,8523
1,S-Learner (Logistic),0.006324,-0.026735,0.036971,0.069996,0.023375,0.113450,0.067225,0.034010,0.099273,0.069110,0.042560,0.096723,8523
2,T-Learner (GBM),-0.000988,-0.038069,0.033018,0.087802,0.028905,0.141980,0.084704,0.046631,0.119795,0.068470,0.041225,0.097580,8523
3,X-Learner (GBM),-0.002480,-0.038713,0.033391,0.107152,0.056434,0.164296,0.096543,0.058904,0.132834,0.091461,0.064557,0.119602,8523


In [9]:
womens_comparison_df = compare_models_on_test(womens_results)
print("--- Womens E-Mail: test-set comparison (sorted by Qini AUC) ---")
display(womens_comparison_df)

--- Womens E-Mail: test-set comparison (sorted by Qini AUC) ---


,model,qini_auc,qini_auc_ci_lower,qini_auc_ci_upper,uplift_at_10pct,uplift_at_10pct_ci_lower,uplift_at_10pct_ci_upper,uplift_at_20pct,uplift_at_20pct_ci_lower,uplift_at_20pct_ci_upper,uplift_at_30pct,uplift_at_30pct_ci_lower,uplift_at_30pct_ci_upper,n_samples
0,X-Learner (GBM),0.076243,0.042203,0.110605,0.078840,0.035545,0.121781,0.073135,0.040564,0.101241,0.075738,0.049790,0.100827,8538
1,S-Learner (GBM),0.071570,0.036617,0.110607,0.068412,0.015332,0.118672,0.065835,0.028962,0.101767,0.069278,0.040099,0.097034,8538
2,T-Learner (GBM),0.056082,0.020204,0.093234,0.079600,0.022193,0.124790,0.068106,0.031699,0.098302,0.066651,0.040499,0.093380,8538
3,S-Learner (Logistic),0.016954,-0.015461,0.050392,0.064916,0.018045,0.117148,0.059663,0.027202,0.089069,0.064031,0.034734,0.087413,8538


The `_ci_lower`/`_ci_upper` columns matter because at this project's per-arm+control test size (`n_samples` ≈8,500), a single Qini AUC point estimate is noisy: two models with different point estimates can still have overlapping CIs, meaning the test set doesn't support calling one better than the other. Any "winning model" claim here is read off the highest point estimate; CI width determines how much weight that claim can bear.

Mens: S-Learner (GBM) has the highest point estimate (Qini AUC 0.012), but every model's CI includes zero (e.g. S-GBM: [-0.021, 0.048]) — no model beats random targeting at this sample size, the ordering is noise.

Womens: X-Learner (GBM) (0.076) and S-Learner (GBM) (0.072) are the top two, both with CIs excluding zero (X: [0.042, 0.111]; S-GBM: [0.037, 0.111]) — genuine uplift signal, but the two CIs overlap broadly, so treat them as tied rather than X-Learner being a clear winner.

## Step 3: Qini curves for the best model per arm

A single Qini AUC number hides *where* in the ranking a model is strong or weak. `plot_qini_curves_comparison` plots the full cumulative-gain curve against random targeting for every model in an arm, with the top model highlighted — Mens below is illustrative only given the CI from Step 2, Womens is the one with actual separation from the random baseline.

In [10]:

_mens_top_row = mens_comparison_df.iloc[0]
_mens_top_name = _mens_top_row["model"]
_mens_ci_lower = _mens_top_row["qini_auc_ci_lower"]
_mens_ci_upper = _mens_top_row["qini_auc_ci_upper"]

if _mens_ci_lower <= 0.0 <= _mens_ci_upper or _mens_ci_upper < 0.0:
    warnings.warn(
        f"Mens E-Mail: top model '{_mens_top_name}' has Qini AUC 95 % CI "
        f"[{_mens_ci_lower:.4f}, {_mens_ci_upper:.4f}] which includes zero. "
        "Ranking quality is statistically indistinguishable from random at "
        "this sample size. Selecting it as the least-bad candidate for "
        "downstream illustration — no statistically distinguishable winner "
        "exists for the Mens segment.",
        UserWarning,
        stacklevel=2,
    )
mens_best_model = _mens_top_name  # least-bad candidate; may not be reliable

# ── Womens: straightforward — select by highest point estimate ───────────
womens_best_model = womens_comparison_df.iloc[0]["model"]

print(f"Mens E-Mail   -- best model on test (point estimate): {mens_best_model}")
print(f"Womens E-Mail -- best model on test: {womens_best_model}")




plot_qini_curves_comparison(
    mens_results,
    mens_comparison_df,
    mens_best_model,
    title="Qini curves (test set) — Mens E-Mail — all models compared",
).show()

plot_qini_curves_comparison(
    womens_results,
    womens_comparison_df,
    womens_best_model,
    title="Qini curves (test set) — Womens E-Mail — all models compared",
).show()

Mens E-Mail   -- best model on test (point estimate): S-Learner (GBM)
Womens E-Mail -- best model on test: X-Learner (GBM)


/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3747: UserWarning: Mens E-Mail: top model 'S-Learner (GBM)' has Qini AUC 95 % CI [-0.0207, 0.0477] which includes zero. Ranking quality is statistically indistinguishable from random at this sample size. Selecting it as the least-bad candidate for downstream illustration — no statistically distinguishable winner exists for the Mens segment.
  exec(code_obj, self.user_global_ns, self.user_ns)


The Womens curve separates visibly from the diagonal (random targeting) across most of the population range, consistent with its CI excluding zero. The Mens curve crosses the diagonal repeatedly and stays close to it throughout — the visual counterpart of a CI that includes zero.

## Step 4: Business-framed targeting impact

`compute_targeting_impact` translates a predicted-uplift ranking into three targeting policies at a given budget fraction `k`: model-guided (top-`k` by predicted uplift), random targeting, and treat-everyone.

`total_population` below is set to this project's full experiment sample (train+val+test, ≈64,000) as a stand-in for the customer base a real campaign would target. In production this should be the size of the *current* active customer file, which won't match the historical experiment's sample size — swap it in before treating this as a real budget estimate.

`value_per_incremental_outcome` stays at its default (`None`), so results are in incremental-visit counts, not revenue. `visit` isn't itself a revenue event (see `methodology.md`, "Modeling Approach"); converting to dollars needs a conversion-given-visit rate and an average order value — deployment-specific numbers that don't belong hardcoded into `src/evaluation.py`.

Mens table below is included only for structural parallelism with Womens — same non-signal as Step 2/3, not evidence of reliable lift.

In [11]:
total_population = len(train_df) + len(val_df) + len(test_df)
print(f"total_population used below (train+val+test size, see assumption note above): {total_population}")

mens_best_predictions = mens_test_predictions[mens_best_model]

targeting_rows = []
for k in (0.10, 0.20, 0.30):
    impact = compute_targeting_impact(
        y_test_mens, mens_best_predictions, treat_test_mens,
        k=k, total_population=total_population,
        value_per_incremental_outcome=None,
        n_bootstrap=1000, ci_level=0.95, random_state=42,
    )
    targeting_rows.append({
        "arm": "Mens E-Mail",
        "model": mens_best_model,
        "k": k,
        "n_targeted": impact["n_targeted"],
        "model_guided_uplift_rate": impact["model_guided_uplift_rate"]["point_estimate"],
        "model_guided_uplift_rate_ci_lower": impact["model_guided_uplift_rate"]["ci_lower"],
        "model_guided_uplift_rate_ci_upper": impact["model_guided_uplift_rate"]["ci_upper"],
        "model_guided_incremental_visits": impact["model_guided_incremental_visits"],
        "random_targeting_incremental_visits": impact["random_targeting_incremental_visits"],
        "treat_everyone_incremental_visits": impact["treat_everyone_incremental_visits"],
    })

mens_targeting_df = pd.DataFrame(targeting_rows)
display(mens_targeting_df)

total_population used below (train+val+test size, see assumption note above): 64000


,arm,model,k,n_targeted,model_guided_uplift_rate,model_guided_uplift_rate_ci_lower,model_guided_uplift_rate_ci_upper,model_guided_incremental_visits,random_targeting_incremental_visits,treat_everyone_incremental_visits
0,Mens E-Mail,S-Learner (GBM),0.1,6400,0.111710,0.053809,0.168581,714.946396,478.871337,4788.713365
1,Mens E-Mail,S-Learner (GBM),0.2,12800,0.085196,0.046602,0.122396,1090.514404,957.742673,4788.713365
2,Mens E-Mail,S-Learner (GBM),0.3,19200,0.075923,0.048627,0.107399,1457.718111,1436.614010,4788.713365


In [12]:
womens_best_predictions = womens_test_predictions[womens_best_model]

targeting_rows = []
for k in (0.10, 0.20, 0.30):
    impact = compute_targeting_impact(
        y_test_womens, womens_best_predictions, treat_test_womens,
        k=k, total_population=total_population,
        value_per_incremental_outcome=None,
        n_bootstrap=1000, ci_level=0.95, random_state=42,
    )
    targeting_rows.append({
        "arm": "Womens E-Mail",
        "model": womens_best_model,
        "k": k,
        "n_targeted": impact["n_targeted"],
        "model_guided_uplift_rate": impact["model_guided_uplift_rate"]["point_estimate"],
        "model_guided_uplift_rate_ci_lower": impact["model_guided_uplift_rate"]["ci_lower"],
        "model_guided_uplift_rate_ci_upper": impact["model_guided_uplift_rate"]["ci_upper"],
        "model_guided_incremental_visits": impact["model_guided_incremental_visits"],
        "random_targeting_incremental_visits": impact["random_targeting_incremental_visits"],
        "treat_everyone_incremental_visits": impact["treat_everyone_incremental_visits"],
    })

womens_targeting_df = pd.DataFrame(targeting_rows)
display(womens_targeting_df)

,arm,model,k,n_targeted,model_guided_uplift_rate,model_guided_uplift_rate_ci_lower,model_guided_uplift_rate_ci_upper,model_guided_incremental_visits,random_targeting_incremental_visits,treat_everyone_incremental_visits
0,Womens E-Mail,X-Learner (GBM),0.1,6400,0.078840,0.035545,0.121781,504.578305,310.309605,3103.096048
1,Womens E-Mail,X-Learner (GBM),0.2,12800,0.073135,0.040564,0.101241,936.129882,620.619210,3103.096048
2,Womens E-Mail,X-Learner (GBM),0.3,19200,0.075738,0.049790,0.100827,1454.178294,930.928814,3103.096048


## Summary

- `test.parquet` was touched for the first time in this notebook, exactly once, for final evaluation only.
- **Mens E-Mail:** no model's ranking is distinguishable from random — every Qini AUC 95% CI includes zero. Do not deploy a Mens targeting model on this evidence; more data or a different feature set would be needed first.
- **Womens E-Mail:** X-Learner (GBM) has the highest point estimate (Qini AUC 0.076, CI excludes zero) and matches `methodology.md`'s prior, but S-Learner (GBM) is statistically indistinguishable from it (0.072, near-identical CI) — both lead, not a clean win between the two. Targeting the top 10–30% by predicted uplift beats random targeting at every tested budget regardless of which of the two is used.
- Business takeaway: a Womens-email campaign targeted by predicted uplift is supported by this evaluation; a Mens-email campaign is not, at least not with the current model and feature set.